#### Classical Machine Learning for Text

- **Goal:** Establish a strong baseline. Understand how to turn text into numbers and the mechanics of classification (Weights & Gradients).
  - **Day 2 & 3 (Binary Classification & Training):** The "Black Box" (Sklearn) vs. The "White Box" (PyTorch). Implementing the manual training loop (Forward, Loss, Backward) and comparing Black Box Logistic Regression, SVM, Tree Model and Simple NN.
- **Sources:**
  - PyTorch Linear Layer: https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html
  - Sklearn models
    - linear models: https://scikit-learn.org/stable/modules/linear_model.html
    - Support Vector Machines: https://scikit-learn.org/stable/modules/svm.html
    - Tree Model: https://scikit-learn.org/stable/modules/tree.html
    - Ensembles: https://scikit-learn.org/stable/modules/ensemble.html
    - metrics: https://scikit-learn.org/stable/api/sklearn.metrics.html

In [3]:
# suppress warnings 
from warnings import filterwarnings
filterwarnings("ignore") 

# import the appropriate libraries
from copy import deepcopy
import torch
from torch import nn
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from utils import load_sklearn_features, load_tensors_and_create_dataloaders

In [4]:
# Load data and recreate DataLoaders using utility for tensorflow models
train_loader, val_loader, test_loader = load_tensors_and_create_dataloaders('datasets/imdb_tensors.pt', batch_size=32)

print("Day 2 Data Loaded Successfully!")
print(f"Train batch count: {len(train_loader)}")

Day 2 Data Loaded Successfully!
Train batch count: 313


In [5]:
# Load data for Scikit-Learn models with sparse matrix
X_train_sp, y_train_sp, X_val_sp, y_val_sp, X_test_sp, y_test_sp = load_sklearn_features('datasets/imdb_sklearn')

## Linear Models 
* **Assumption:** Assumes a linear relationship between the feature vector and labels. It also assumes linear independence among features (little to no multicollinearity).
    $$ \text{Y} = \text{X}W^T + \text{b} $$

### PyTorch Implementation 
- **Linear Regressor:** Outputs a continuous value (raw score).
- **Linear Classifier:** Outputs a probability from 0 to 1. 
  - Uses the **Sigmoid** activation function to squash the raw output:
    $$ \sigma(z) = \frac{1}{1 + e^{-z}} $$

### Loss Function
We use **Binary Cross Entropy** (BCE):
$$ \mathcal{L} = - \left( y \cdot \log(\hat{y}) + (1 - y) \cdot \log(1 - \hat{y}) \right) $$

**Implementation Note:** 
PyTorch's `nn.BCEWithLogitsLoss` applies the Sigmoid function internally. 
*   **Do not** add `nn.Sigmoid` to the final layer of your model.
*   The model should output raw **logits** ($z$), and the loss function will convert them to probabilities ($\hat{y}$) during calculation. This provides better numerical stability.


## Linear Models 
* **Assumption:** Assumes a linear relationship between the feature vector and labels. It also assumes linear independence among features (little to no multicollinearity).
    $$ \text{Y} = \text{X}W^T + \text{b} $$

### PyTorch Implementation 
- **Linear Regressor:** Outputs a continuous value (raw score).
- **Linear Classifier:** Outputs a probability from 0 to 1. 
  - Uses the **Sigmoid** activation function to squash the raw output:
    $$ \sigma(z) = \frac{1}{1 + e^{-z}} $$

### Loss Function
We use **Binary Cross Entropy** (BCE):
$$ \mathcal{L} = - \left( y \cdot \log(\hat{y}) + (1 - y) \cdot \log(1 - \hat{y}) \right) $$

**Implementation Note:** 
PyTorch's `nn.BCEWithLogitsLoss` applies the Sigmoid function internally. 
*   **Do not** add `nn.Sigmoid` to the final layer of your model.
*   The model should output raw **logits** ($z$), and the loss function will convert them to probabilities ($\hat{y}$) during calculation. This provides better numerical stability.

#### **Why is this more stable? (The "Log-Sum-Exp" Trick)**
If you use `Sigmoid` followed by `BCELoss`, you risk numerical instability.
1.  **The Problem:** If the model is very confident but wrong (e.g., outputs -100), $\sigma(-100)$ becomes effectively $0.0$ due to floating-point precision. The loss formula then tries to calculate $\log(0)$, which is $-\infty$ (NaN). This crashes training.
2.  **The Solution:** `BCEWithLogitsLoss` combines the operations into a single mathematical step. It uses the **Log-Sum-Exp** trick to simplify the formula analytically before calculating, ensuring the computer never has to evaluate $\log(0)$ or $\exp(\text{huge number})$.
Instead it sees $-\log(1 + e^{100})$. For large numbers, $\log(1 + x) \approx \log(x)$ so it simplifies to $-\log(e^{100})$ which equals -100.



In [6]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
class LinearModel(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.linear_layer = nn.Linear(in_features=feature_dim, out_features=1) # either a score or a 0 or 1 via sigmoid 

    def forward(self, X):
        return self.linear_layer(X) 

def check_early_stop(patience, val_loss, model, best_model_state, lowest_loss, counter, epoch):
    early_stop = False
    if val_loss < lowest_loss:
        lowest_loss = val_loss
        best_model_state = deepcopy(model.state_dict())
        counter = 0 
    else:
        counter += 1

    if counter == patience:
        early_stop = True
    return best_model_state, lowest_loss, counter, early_stop

def get_total_accuracy(logits, y, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    return (preds == y).float().sum().item()

def evaluate_model(model, data_loader, criterion, device) -> tuple:
    if len(data_loader) <= 0: return 0.0, 0.0
    model.eval()
    total_size = len(data_loader.dataset)
    avg_loss = 0 
    accuracy = 0 
    with torch.no_grad():
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            logit = model(x)
            loss = criterion(logit, y)
            avg_loss += loss.item()
            accuracy += get_total_accuracy(logit, y)
    avg_loss/=len(data_loader)
    accuracy/=total_size
    return avg_loss, accuracy

def train_test_model(device, model, criterion, train_loader, val_loader, patience, num_epochs, lr):
    model.to(device)
    optimizer = torch.optim.Adam(params=model.parameters(), lr=lr)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer, step_size=3, gamma=0.5)
    best_model_state = deepcopy(model.state_dict())
    lowest_loss = float("inf")
    counter = 0 
    total_train_size = len(train_loader.dataset)
    for epoch in range(num_epochs):
        model.train()
        avg_train_loss = 0 
        train_accuracy = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logit = model(x)
            loss = criterion(logit, y)
            loss.backward()
            optimizer.step()
            avg_train_loss += loss.item()
            train_accuracy += get_total_accuracy(logit, y)
        avg_train_loss /= len(train_loader)
        train_accuracy /= total_train_size

        avg_val_loss, val_accuracy = evaluate_model(model, val_loader, criterion, device)
        lr_scheduler.step()
        best_model_state, lowest_loss, counter, early_stop = check_early_stop(
            patience, avg_val_loss, model, best_model_state, lowest_loss, counter, epoch)
        if epoch % 10 == 0: 
            print(f"Epoch number {epoch} train_loss: {avg_train_loss:.4f} and val_loss: {avg_val_loss:.4f}")
            print(f"Epoch number {epoch} train_accuracy: {train_accuracy:.4f} and val_accuracy: {val_accuracy:.4f}")
        if early_stop:
            print(f"Early stoping with epoch number {epoch}")
            print(f"Train_loss: {avg_train_loss:.4f} and val_loss: {avg_val_loss:.4f}")
            break
    model.load_state_dict(best_model_state)
    return model 

In [7]:
linear_classifier = LinearModel(2000)
criterion = nn.BCEWithLogitsLoss() # this applies sigmoid activation so you don't need to us nn.Sigmoid layer
patience = 10
num_epochs = 50
lr = 1e-3
train_classifier = train_test_model(
    device,
    linear_classifier,
    criterion,
    train_loader,
    val_loader,
    patience,
    num_epochs,
    lr
)
test_loss, test_accuracy = evaluate_model(train_classifier, test_loader, criterion, device)
print(f"test_loss: {test_loss:.4f} and test_accuracy: {test_accuracy:.4f}")

Epoch number 0 train_loss: 0.6702 and val_loss: 0.6477
Epoch number 0 train_accuracy: 0.7727 and val_accuracy: 0.8150
Epoch number 10 train_loss: 0.5027 and val_loss: 0.5187
Epoch number 10 train_accuracy: 0.8602 and val_accuracy: 0.8300
Epoch number 20 train_loss: 0.4909 and val_loss: 0.5094
Epoch number 20 train_accuracy: 0.8608 and val_accuracy: 0.8300
Epoch number 30 train_loss: 0.4897 and val_loss: 0.5085
Epoch number 30 train_accuracy: 0.8610 and val_accuracy: 0.8300
Epoch number 40 train_loss: 0.4896 and val_loss: 0.5084
Epoch number 40 train_accuracy: 0.8610 and val_accuracy: 0.8300
test_loss: 0.4823 and test_accuracy: 0.8960


## Test using sklearn models and accuracy_score

In [9]:
random_state = 204
sklearn_models = {
    "logistic regression model": LogisticRegression(random_state=random_state, max_iter=1000),
    "support vector machine": LinearSVC(random_state=random_state),
    "decision tree": DecisionTreeClassifier(random_state=random_state),
    "random forest": RandomForestClassifier(random_state=random_state),
    "gradient boosted tree": GradientBoostingClassifier(random_state=random_state)
}

def train_test_sklearn(model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred) 

for model_name, model in sklearn_models.items():
    print(f"Testing model: {model_name}")
    metric = train_test_sklearn(model, X_train_sp, y_train_sp, X_val_sp, y_val_sp)
    print(f"validation accuracy: {metric:.4f}")

Testing model: logistic regression model
validation accuracy: 0.8350
Testing model: support vector machine
validation accuracy: 0.8400
Testing model: decision tree
validation accuracy: 0.7150
Testing model: random forest
validation accuracy: 0.8000
Testing model: gradient boosted tree
validation accuracy: 0.7800


In [11]:
# --- 1. Logistic Regression Tuning ---
print("--- Tuning Logistic Regression ---")
best_lr_acc = 0
best_lr_C = 0
# C is inverse regularization strength (smaller = stronger reg)
for C in [0.01, 0.1, 1, 10]:
    clf = LogisticRegression(C=C, max_iter=1000, random_state=42)
    clf.fit(X_train_sp, y_train_sp)
    
    val_pred = clf.predict(X_val_sp)
    acc = accuracy_score(y_val_sp, val_pred)
    print(f"C={C}: Val Acc = {acc:.4f}")
    
    if acc > best_lr_acc:
        best_lr_acc = acc
        best_lr_C = C

print(f"🏆 Best Logistic Regression: C={best_lr_C} (Acc: {best_lr_acc:.4f})")


# --- 2. Linear SVM (LinearSVC) Tuning ---
# Fast, good for high-dimensional sparse text
print("\n--- Tuning Linear SVM ---")
best_lsvm_acc = 0
best_lsvm_C = 0
for C in [0.01, 0.1, 1, 10]:
    svm_clf = LinearSVC(C=C, dual="auto", random_state=42)
    svm_clf.fit(X_train_sp, y_train_sp)
    
    val_pred = svm_clf.predict(X_val_sp)
    acc = accuracy_score(y_val_sp, val_pred)
    print(f"C={C}: Val Acc = {acc:.4f}")
    
    if acc > best_lsvm_acc:
        best_lsvm_acc = acc
        best_lsvm_C = C

print(f"🏆 Best Linear SVM: C={best_lsvm_C} (Acc: {best_lsvm_acc:.4f})")


# --- 3. Decision Tree Tuning ---
print("\n--- Tuning Decision Tree ---")
best_dt_acc = 0
best_depth = 0
for depth in [10, 20, 50, None]: 
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_sp, y_train_sp)
    
    val_pred = dt.predict(X_val_sp)
    acc = accuracy_score(y_val_sp, val_pred)
    print(f"Max Depth={depth}: Val Acc = {acc:.4f}")
    
    if acc > best_dt_acc:
        best_dt_acc = acc
        best_depth = depth

print(f"🏆 Best Tree: Depth={best_depth} (Acc: {best_dt_acc:.4f})")


# --- 4. Random Forest Tuning ---
print("\n--- Tuning Random Forest ---")
best_rf_acc = 0
best_rf_est = 0
# n_estimators = number of trees
for n_est in [50, 100, 200]:
    rf = RandomForestClassifier(n_estimators=n_est, max_depth=20, random_state=42)
    rf.fit(X_train_sp, y_train_sp)
    
    val_pred = rf.predict(X_val_sp)
    acc = accuracy_score(y_val_sp, val_pred)
    print(f"Trees={n_est}: Val Acc = {acc:.4f}")
    
    if acc > best_rf_acc:
        best_rf_acc = acc
        best_rf_est = n_est

print(f"🏆 Best Random Forest: Trees={best_rf_est} (Acc: {best_rf_acc:.4f})")


# --- 5. Gradient Boosting Tuning ---
print("\n--- Tuning Gradient Boosting ---")
best_gb_acc = 0
best_gb_lr = 0
# learning_rate controls how strongly each tree corrects the errors of the previous ones
for lr in [0.01, 0.1, 0.2]:
    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, max_depth=3, random_state=42)
    gb.fit(X_train_sp, y_train_sp)
    
    val_pred = gb.predict(X_val_sp)
    acc = accuracy_score(y_val_sp, val_pred)
    print(f"LR={lr}: Val Acc = {acc:.4f}")
    
    if acc > best_gb_acc:
        best_gb_acc = acc
        best_gb_lr = lr

print(f"🏆 Best Gradient Boosting: LR={best_gb_lr} (Acc: {best_gb_acc:.4f})")



--- Tuning Logistic Regression ---
C=0.01: Val Acc = 0.7600
C=0.1: Val Acc = 0.8150
C=1: Val Acc = 0.8350
C=10: Val Acc = 0.8350
🏆 Best Logistic Regression: C=1 (Acc: 0.8350)

--- Tuning Linear SVM ---
C=0.01: Val Acc = 0.8100
C=0.1: Val Acc = 0.8350
C=1: Val Acc = 0.8400
C=10: Val Acc = 0.8300
🏆 Best Linear SVM: C=1 (Acc: 0.8400)

--- Tuning Decision Tree ---
Max Depth=10: Val Acc = 0.6950
Max Depth=20: Val Acc = 0.7150
Max Depth=50: Val Acc = 0.7300
Max Depth=None: Val Acc = 0.7400
🏆 Best Tree: Depth=None (Acc: 0.7400)

--- Tuning Random Forest ---
Trees=50: Val Acc = 0.7900
Trees=100: Val Acc = 0.8000
Trees=200: Val Acc = 0.7950
🏆 Best Random Forest: Trees=100 (Acc: 0.8000)

--- Tuning Gradient Boosting ---
LR=0.01: Val Acc = 0.6550
LR=0.1: Val Acc = 0.7800
LR=0.2: Val Acc = 0.8100
🏆 Best Gradient Boosting: LR=0.2 (Acc: 0.8100)
